# Introduction to Nested Learning

Welcome to Nested Learning! This notebook introduces the core concepts and demonstrates basic usage.

## What is Nested Learning?

Nested Learning reframes machine learning as a system of **nested optimization problems** where different parameters are updated at different frequencies:

- **Fast parameters** (every step): Quick adaptation to current data
- **Medium parameters** (every 10 steps): General feature learning
- **Slow parameters** (every 100 steps): Stable, fundamental representations

This mimics biological learning where different brain regions adapt at different timescales.

## Key Benefits

1. **Prevents catastrophic forgetting** in continual learning
2. **Better generalization** through stable low-level features
3. **Efficient learning** by separating fast/slow components
4. **Improved long-term stability** in training

## Setup

First, let's import the necessary libraries and modules.

In [ ]:
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.models import NestedMLP
from src.optimizers import NestedOptimizerBuilder
from src.training import NestedTrainer

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✓ All imports successful!")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## Part 1: Simple Comparison

Let's compare standard training with Nested Learning on a simple task.

In [ ]:
# Generate synthetic dataset
def generate_data(n_samples=500, n_features=10, n_classes=3):
    """Generate simple classification dataset."""
    X = torch.randn(n_samples, n_features)
    
    # Create non-linear decision boundary
    y = (X[:, 0]**2 + X[:, 1]**2 > 1).long()
    y += (X[:, 2] > 0).long()
    y = y % n_classes
    
    return X, y

# Generate train and test data
X_train, y_train = generate_data(500)
X_test, y_test = generate_data(200)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {torch.unique(y_train).tolist()}")

### Standard Training

In [ ]:
# Standard model
standard_model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

standard_optimizer = torch.optim.Adam(standard_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Train
standard_losses = []
standard_model.train()

for epoch in range(100):
    standard_optimizer.zero_grad()
    outputs = standard_model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    standard_optimizer.step()
    
    standard_losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

# Evaluate
standard_model.eval()
with torch.no_grad():
    test_outputs = standard_model(X_test)
    test_acc = (test_outputs.argmax(1) == y_test).float().mean()
    
print(f"\n✓ Standard model test accuracy: {test_acc:.4f}")

### Nested Learning Training

In [ ]:
# Nested model
nested_model = NestedMLP(
    input_dim=10,
    hidden_dims=[32, 16],
    output_dim=3
)

# Build multi-frequency optimizer
builder = NestedOptimizerBuilder(nested_model, num_levels=3)
builder.auto_assign_params('uniform')
nested_optimizer = builder.build(
    optimizer_types=['adam', 'sgd', 'sgd'],
    learning_rates=[0.01, 0.05, 0.1],
    frequencies=[1, 5, 25]  # Fast, Medium, Slow
)

print("✓ Nested optimizer created with 3 frequency levels")
print(f"  - Fast (f=1): Updates every step")
print(f"  - Medium (f=5): Updates every 5 steps")
print(f"  - Slow (f=25): Updates every 25 steps")

In [ ]:
# Train with Nested Learning
nested_losses = []
nested_model.train()

for epoch in range(100):
    nested_optimizer.zero_grad()
    outputs = nested_model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    nested_optimizer.step(step=epoch)  # Pass step for frequency-aware updates
    
    nested_losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

# Evaluate
nested_model.eval()
with torch.no_grad():
    test_outputs = nested_model(X_test)
    test_acc = (test_outputs.argmax(1) == y_test).float().mean()
    
print(f"\n✓ Nested Learning test accuracy: {test_acc:.4f}")

### Compare Training Dynamics

In [ ]:
# Plot comparison
plt.figure(figsize=(12, 4))

# Training loss
plt.subplot(1, 2, 1)
plt.plot(standard_losses, 'r--', label='Standard', linewidth=2, alpha=0.7)
plt.plot(nested_losses, 'b-', label='Nested Learning', linewidth=2, alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Loss difference
plt.subplot(1, 2, 2)
loss_diff = np.array(standard_losses) - np.array(nested_losses)
plt.plot(loss_diff, 'g-', linewidth=2)
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Epoch')
plt.ylabel('Loss Difference (Standard - Nested)')
plt.title('Loss Difference Over Time')
plt.grid(True, alpha=0.3)
plt.fill_between(range(len(loss_diff)), 0, loss_diff, alpha=0.3, color='g')

plt.tight_layout()
plt.show()

print(f"\nFinal loss - Standard: {standard_losses[-1]:.4f}, Nested: {nested_losses[-1]:.4f}")
print(f"Average improvement: {np.mean(loss_diff[20:]):.4f}")

## Part 2: Understanding Multi-Frequency Updates

Let's visualize how different parameter groups update at different frequencies.

In [ ]:
# Track which levels update at each step
from src.training.metrics import UpdateFrequencyTracker

tracker = UpdateFrequencyTracker(num_levels=3)

# Simulate 100 steps
for step in range(100):
    # Determine which levels should update
    active_levels = []
    if step % 1 == 0:  # Fast
        active_levels.append(0)
    if step % 5 == 0:  # Medium
        active_levels.append(1)
    if step % 25 == 0:  # Slow
        active_levels.append(2)
    
    tracker.record_update(step, active_levels)

# Get statistics
stats = tracker.get_statistics()
print("Update Statistics (100 steps):")
for level, count in stats['total_updates'].items():
    freq = ['Fast', 'Medium', 'Slow'][level]
    print(f"  Level {level} ({freq}): {count} updates")

In [ ]:
# Visualize update patterns
steps = list(range(100))
fast_updates = [1 if step % 1 == 0 else 0 for step in steps]
medium_updates = [1 if step % 5 == 0 else 0 for step in steps]
slow_updates = [1 if step % 25 == 0 else 0 for step in steps]

plt.figure(figsize=(14, 6))

plt.subplot(3, 1, 1)
plt.bar(steps[:50], fast_updates[:50], color='blue', alpha=0.7)
plt.ylabel('Fast\n(every step)')
plt.title('Multi-Frequency Update Pattern (First 50 Steps)')
plt.ylim([0, 1.2])
plt.yticks([0, 1])

plt.subplot(3, 1, 2)
plt.bar(steps[:50], medium_updates[:50], color='orange', alpha=0.7)
plt.ylabel('Medium\n(every 5 steps)')
plt.ylim([0, 1.2])
plt.yticks([0, 1])

plt.subplot(3, 1, 3)
plt.bar(steps[:50], slow_updates[:50], color='red', alpha=0.7)
plt.ylabel('Slow\n(every 25 steps)')
plt.xlabel('Training Step')
plt.ylim([0, 1.2])
plt.yticks([0, 1])

plt.tight_layout()
plt.show()

## Part 3: Parameter Assignment Strategies

Different strategies for assigning parameters to frequency levels.

In [ ]:
# Create a model
model = NestedMLP(input_dim=10, hidden_dims=[64, 32], output_dim=5)

# Strategy 1: Uniform (default)
builder1 = NestedOptimizerBuilder(model, num_levels=3)
builder1.auto_assign_params('uniform')
counts1 = [len(builder1.param_groups[i]) for i in range(3)]

# Strategy 2: Alternating
builder2 = NestedOptimizerBuilder(model, num_levels=3)
builder2.auto_assign_params('alternating')
counts2 = [len(builder2.param_groups[i]) for i in range(3)]

# Strategy 3: Depth-based (manual)
builder3 = NestedOptimizerBuilder(model, num_levels=3)
# Input layer -> Slow (level 2)
# Hidden layers -> Medium (level 1)
# Output layer -> Fast (level 0)
params_by_depth = model.get_parameters_by_frequency()
for level, params in params_by_depth.items():
    for param in params:
        builder3.assign_parameter(param, level)
counts3 = [len(builder3.param_groups[i]) for i in range(3)]

# Visualize
strategies = ['Uniform', 'Alternating', 'Depth-based']
x = np.arange(3)
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, counts1, width, label='Uniform', alpha=0.7)
plt.bar(x, counts2, width, label='Alternating', alpha=0.7)
plt.bar(x + width, counts3, width, label='Depth-based', alpha=0.7)
plt.xlabel('Frequency Level')
plt.ylabel('Number of Parameters')
plt.title('Parameter Assignment Strategies')
plt.xticks(x, ['Fast (L0)', 'Medium (L1)', 'Slow (L2)'])
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Parameter counts by strategy:")
for strategy, counts in zip(strategies, [counts1, counts2, counts3]):
    print(f"  {strategy}: Fast={counts[0]}, Medium={counts[1]}, Slow={counts[2]}")

## Key Takeaways

1. **Multi-frequency updates**: Different parameters adapt at different rates
2. **Training dynamics**: Often smoother and more stable
3. **Flexibility**: Multiple assignment strategies available
4. **Simple API**: Easy to integrate into existing PyTorch code

## Next Steps

- **Notebook 02**: Deep dive into multi-frequency training
- **Notebook 03**: Continuum Memory System
- **Notebook 04**: Meta-learning with DMGD
- **Notebook 05**: Hope architecture for language modeling

---

**Try modifying the code above!**
- Change frequency values: `[1, 5, 25]` → `[1, 10, 100]`
- Try different architectures
- Experiment with learning rates